In [1]:
# =====================================
# CONSTRUCCIÓN DE LA CAPA GOLD
# Consolidación de restaurantes enriquecidos en tabla única
# =====================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

# Rutas
PROCESSED_DIR = Path("../data/processed")
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas y rutas configuradas")
print(f"Processed: {PROCESSED_DIR.resolve()}")
print(f"Gold: {GOLD_DIR.resolve()}")

Librerías cargadas y rutas configuradas
Processed: /Users/juana/Desktop/tfm-data-science-gtm/data/processed
Gold: /Users/juana/Desktop/tfm-data-science-gtm/data/gold


In [2]:
# =====================================
# CARGAR FICHEROS DE PROCESSED
# =====================================

# Los tres ficheros que guardamos en la sesión anterior
enriq_direccion = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_direccion.parquet")
enriq_proximidad = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_proximidad.parquet")
censo_rest = pd.read_parquet(PROCESSED_DIR / "censo_restaurantes_madrid.parquet")

print("Ficheros cargados:")
print(f"  Enriquecidos por dirección:  {len(enriq_direccion)} filas × {len(enriq_direccion.columns)} columnas")
print(f"  Enriquecidos por proximidad: {len(enriq_proximidad)} filas × {len(enriq_proximidad.columns)} columnas")
print(f"  Censo restaurantes:          {len(censo_rest)} filas × {len(censo_rest.columns)} columnas")

Ficheros cargados:
  Enriquecidos por dirección:  818 filas × 70 columnas
  Enriquecidos por proximidad: 737 filas × 27 columnas
  Censo restaurantes:          1626 filas × 52 columnas


In [3]:
# =====================================
# INSPECCIONAR COLUMNAS DE CADA FUENTE
# =====================================

print("=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===")
print(list(enriq_direccion.columns))

print("\n=== ENRIQUECIDOS POR PROXIMIDAD (columnas) ===")
print(list(enriq_proximidad.columns))

=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===
['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine', 'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city', 'phone', 'website', 'opening_hours', 'outdoor_seating', 'takeaway', 'delivery', 'wheelchair', 'calle_norm_osm', 'numero_norm_osm', 'clave_cruce', 'id_local', 'id_distrito_local', 'desc_distrito_local', 'id_barrio_local', 'desc_barrio_local', 'cod_barrio_local', 'id_seccion_censal_local', 'desc_seccion_censal_local', 'coordenada_x_local', 'coordenada_y_local', 'id_tipo_acceso_local', 'desc_tipo_acceso_local', 'id_situacion_local', 'desc_situacion_local', 'id_vial_edificio', 'clase_vial_edificio', 'desc_vial_edificio', 'id_ndp_edificio', 'id_clase_ndp_edificio', 'nom_edificio', 'num_edificio', 'cal_edificio', 'secuencial_local_PC', 'id_vial_acceso', 'clase_vial_acceso', 'desc_vial_acceso', 'id_ndp_acceso', 'id_clase_ndp_acceso', 'nom_acceso', 'num_acceso', 'cal_acceso', 'coordenada_x_agrupacion', 'coordenada_y_agrupacion', 

In [4]:
# =====================================
# UNIFICAR TABLAS ENRIQUECIDAS
# =====================================

# Columnas base de OSM (están en ambas tablas)
cols_osm = ['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine',
            'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city',
            'phone', 'website', 'opening_hours', 'outdoor_seating',
            'takeaway', 'delivery', 'wheelchair']

# Columnas del censo (están en ambas, aunque proximidad tiene menos)
cols_censo = ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
              'id_seccion_censal_local', 'metodo_match']

# La tabla de dirección tiene además el tipo de acceso; la de proximidad no
# Añadimos tipo de acceso solo si existe
cols_direccion = cols_osm + cols_censo
if 'desc_tipo_acceso_local' in enriq_direccion.columns:
    cols_direccion = cols_direccion + ['desc_tipo_acceso_local']

# Preparar cada tabla con las columnas comunes
tabla_dir = enriq_direccion[cols_direccion].copy()

tabla_prox = enriq_proximidad[cols_osm + cols_censo].copy()
tabla_prox['desc_tipo_acceso_local'] = np.nan  # no disponible en proximidad

# Unir las dos
enriquecidos = pd.concat([tabla_dir, tabla_prox], ignore_index=True)

print(f"Total restaurantes enriquecidos unificados: {len(enriquecidos)}")
print(f"  Por dirección:  {(enriquecidos['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad: {(enriquecidos['metodo_match'] == 'proximidad').sum()}")
print(f"\nColumnas: {len(enriquecidos.columns)}")

Total restaurantes enriquecidos unificados: 1555
  Por dirección:  818
  Por proximidad: 737

Columnas: 23
